# 04 — Silver: enrich access events with shortcut target

Joins bronze events (where `isShortcut = true`) against `dim_shortcut_map` (current snapshot) to resolve each access to its underlying target. Native (non-shortcut) events are excluded from this join.

In [1]:
import json, os
from pyspark.sql import functions as F

# When run inside Fabric, the notebook resource folder contains config.json.
# Fabric exposes notebook-attached files via mssparkutils / notebookutils.
try:
    import notebookutils  # type: ignore
    cfg_path = notebookutils.nbResPath + '/builtin/config.json'
    if not os.path.exists(cfg_path):
        # Fallback: lakehouse Files/config.json
        cfg_path = '/lakehouse/default/Files/config.json'
except Exception:
    cfg_path = './config.json'

with open(cfg_path, 'r', encoding='utf-8') as f:
    CFG = json.load(f)

OBS_WS  = CFG['observability_workspace_name']
OBS_LH  = CFG['observability_lakehouse_name']
TBL     = CFG['tables']
API     = CFG['fabric_api']
# monitored_workspaces is a list of workspace display names (strings).
# Backwards-compat: also accept the old [{workspace_name: ...}] shape.
_raw_mon = CFG['monitored_workspaces']
MONITOR = [m if isinstance(m, str) else m['workspace_name'] for m in _raw_mon]
INGEST  = CFG['ingestion']
print(f'Observability workspace : {OBS_WS}')
print(f'Observability lakehouse : {OBS_LH}')
print(f'Monitored workspaces    : {MONITOR}')

StatementMeta(, 1e21eb66-5307-4a77-9089-ac7da7c61644, 3, Finished, Available, Finished, False)

Observability workspace : WS_OnelakeObservability
Observability lakehouse : lh_OnelakeObservability
Monitored workspaces    : ['WS_SagarFabric01', 'WS_SagarFabric03']


In [2]:
from pyspark.sql import functions as F

bronze = spark.table(TBL['bronze'])
dim    = spark.table(TBL['dim_shortcuts'])

# Extract <workspaceItemPrefix> from accessedViaResource. The shortcut is stored
# under <ItemName>.<ItemType>/<path>/<name>, so we strip the leading <Item.Type>/
# and compare against shortcut_full_path.
evt = (bronze
    .where(F.col('isShortcut') == True)
    .withColumn('accessed_path_no_item',
        F.regexp_replace(F.col('accessedViaResource'), r'^[^/]+/', ''))
)

# Join against the current dim snapshot.
# Match d -> e by composing the dim side as
#   <source_workspace_id>/<source_item_id><shortcut_full_path>
# and checking the bronze event's accessed_path_no_item starts with it.
j = (evt.alias('e').join(
        dim.alias('d'),
        F.col('e.accessed_path_no_item').startswith(
            F.concat(F.lit('/'), F.col('d.source_workspace_id'), F.lit('/'),
                     F.col('d.source_item_id'),
                     F.col('d.shortcut_full_path'))),
        'left'
    ))

enriched = j.select(
    F.col('e.event_date'),
    F.col('e.source_workspace_name').alias('event_emitting_workspace'),
    F.col('e.executingUPN'),
    F.col('e.executingPrincipalType'),
    F.col('e.executingPrincipalId'),
    F.col('e.callerIPAddress'),
    F.col('e.originatingApp'),
    F.col('e.accessStartTime'),
    F.col('e.accessEndTime'),
    F.col('e.operationName'),
    F.col('e.operationCategory'),
    F.col('e.serviceEndpoint'),
    F.col('e.httpStatusCode'),
    F.col('e.contentLength').alias('bytes'),
    F.col('e.correlationId'),
    F.col('e.isShortcut'),
    F.col('e.accessedViaResource').alias('shortcut_path_consumer'),
    F.col('e.Resource').alias('resolved_target_resource'),
    F.col('d.source_workspace_name').alias('shortcut_consumer_workspace'),
    F.col('d.source_item_name').alias('shortcut_consumer_lakehouse'),
    F.col('d.target_type'),
    F.col('d.target_workspace_id'),
    F.col('d.target_workspace_name'),
    F.col('d.target_item_id'),
    F.col('d.target_item_name'),
    F.col('d.target_path'),
    F.col('d.target_connection_id'),
    F.col('d.target_location'),
    F.col('d.target_subpath'),
)

# overwrite — silver is a derived view; recomputing is cheap and keeps logic simple
(enriched.write.format('delta')
    .mode('overwrite').option('overwriteSchema','true')
    .saveAsTable(TBL['silver']))

print('Silver rebuilt. Sample:')
spark.table(TBL['silver']).orderBy(F.col('accessStartTime').desc()).show(10, truncate=False)

StatementMeta(, 1e21eb66-5307-4a77-9089-ac7da7c61644, 4, Finished, Available, Finished, False)

Silver rebuilt. Sample:
+----------+------------------------+-------------------------------------------+----------------------+------------------------------------+---------------+--------------+-------------------+-------------------+-----------------+-----------------+---------------+--------------+-----+------------------------------------+----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------+---------------------------+-----------+-------------------+---------------------+--------------+----------------+-----------+--------------------+------------